# Scientific Reports successor — Step 7

Complete crossed susceptibility matrices and controlled waveform experiments. This notebook consumes Step 6 and stops before reduced-law fitting.

In [ ]:
from pathlib import Path
import subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'successor/scirep-waveform-susceptibility'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_root = Path('/content/picoNewton')
    if not repo_root.exists():
        subprocess.run(['git','clone','https://github.com/khalid-saqr/picoNewton.git',str(repo_root)],check=True)
    subprocess.run(['git','-C',str(repo_root),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(repo_root),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
    study_root = Path('/content/drive/MyDrive/picoNewton_susceptibility')
else:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root/'picoNewton_v3').exists():
        repo_root = repo_root.parent
    study_root = repo_root/'piconewton_susceptibility_outputs'
print({'repo_root':str(repo_root),'study_root':str(study_root),'colab':IN_COLAB})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'picoNewton_v3')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'piconewton_susceptibility')],check=True)

## Resolve prior gates

A clean Drive can reconstruct Steps 2–6. Existing passing outputs are reused only after their validators inspect them.

In [ ]:
step2_root = study_root/'bootstrap'/'step2'
step3_root = study_root/'step3_parent_continuity'
step4_root = study_root/'step4_perturbation'
step5_root = study_root/'step5_harmonic_kernel'
step6_root = study_root/'step6_susceptibility'
step7_root = study_root/'step7_waveform_experiments'
if not (step2_root/'completion_gate.json').exists():
    subprocess.run(['piconewton-susceptibility-bootstrap','--repo-root',str(repo_root),'--storage','local','--local-root',str(study_root)],check=True)
if not (step3_root/'step3_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step3','--step2-root',str(step2_root),'--output',str(step3_root),'--profile','publication'],check=True)
if not (step4_root/'step4_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step4','--step3-root',str(step3_root),'--output',str(step4_root),'--profile','publication'],check=True)
if not (step5_root/'step5_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step5','--step4-root',str(step4_root),'--output',str(step5_root),'--profile','publication'],check=True)
if not (step6_root/'step6_manifest.json').exists():
    subprocess.run(['piconewton-susceptibility-step6','--step5-root',str(step5_root),'--step4-root',str(step4_root),'--output',str(step6_root),'--profile','publication'],check=True)
print({'step6_root':str(step6_root),'step7_root':str(step7_root)})

## Execute Step 7 publication profile

In [ ]:
subprocess.run(['piconewton-susceptibility-step7','--step6-root',str(step6_root),'--output',str(step7_root),'--profile','publication'],check=True)

In [ ]:
import json, pandas as pd
manifest = json.loads((step7_root/'step7_manifest.json').read_text())
assert manifest['status'] == 'complete'
assert manifest['allowed_next_step'] == 8
matrices = pd.read_csv(step7_root/'crossed_susceptibility.csv')
decomposition = pd.read_csv(step7_root/'crossed_variance_decomposition.csv')
display(matrices.query('native_diagonal').loc[:, ['matrix_type','vessel_name','waveform_name','phi_rms']])
display(decomposition)
manifest